<a href="https://colab.research.google.com/github/MallikaReddy-hub/prj495-mental-chatbot/blob/main/training/03_finetune_qlora_and_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft trl datasets sentencepiece
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.2/927.2 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.
Mounted at /content/drive


In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL = "Qwen/Qwen2.5-3B-Instruct"
MAX_STEPS = 20          # quick test first; change to -1 for the full run
OUT = "/content/drive/MyDrive/mental_llm_ckpt"

data = load_dataset("json", data_files={"train": "/content/drive/MyDrive/mh_train.jsonl",
                                        "val":   "/content/drive/MyDrive/mh_val.jsonl"})
data = data.remove_columns(["lang"])

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map="auto", torch_dtype=torch.float16)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

cfg = SFTConfig(output_dir=OUT, num_train_epochs=2, max_steps=MAX_STEPS,
                per_device_train_batch_size=1, gradient_accumulation_steps=16,
                learning_rate=2e-4, fp16=True, bf16=False, gradient_checkpointing=True,
                optim="paged_adamw_8bit", logging_steps=5, max_length=1024,
                save_strategy="no", report_to="none")

trainer = SFTTrainer(model=model, args=cfg, train_dataset=data["train"],
                     peft_config=lora, processing_class=tok)

# make the trainable LoRA weights float32 so fp16 training works on the T4
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

trainer.train()
trainer.save_model(OUT + "/final")
print("saved")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Step,Training Loss
5,1.724080
10,1.429460
15,1.282027
20,1.394419


saved


In [ ]:
import torch, os, glob
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUT = "/content/drive/MyDrive/mental_llm_ckpt_full"

data = load_dataset("json", data_files={"train": "/content/drive/MyDrive/mh_train.jsonl",
                                        "val":   "/content/drive/MyDrive/mh_val.jsonl"})
data = data.remove_columns(["lang"])

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map="auto", dtype=torch.float16)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

cfg = SFTConfig(output_dir=OUT, num_train_epochs=2,
                per_device_train_batch_size=1, per_device_eval_batch_size=1,
                gradient_accumulation_steps=16, learning_rate=2e-4,
                fp16=True, bf16=False, gradient_checkpointing=True,
                optim="paged_adamw_8bit", logging_steps=10, max_length=1024,
                eval_strategy="epoch",
                save_strategy="steps", save_steps=40, save_total_limit=2,
                report_to="none")

trainer = SFTTrainer(model=model, args=cfg, train_dataset=data["train"],
                     eval_dataset=data["val"], peft_config=lora, processing_class=tok)

# keep trainable LoRA weights in float32 so fp16 training works on the T4
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

has_ckpt = len(glob.glob(OUT + "/checkpoint-*")) > 0
trainer.train(resume_from_checkpoint=True if has_ckpt else None)

trainer.save_model(OUT + "/final")
tok.save_pretrained(OUT + "/final")
print("saved")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Tokenizing eval dataset:   0%|          | 0/101 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/101 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/101 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/101 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.242091,1.737100,1.709768,690181.000000,0.593656
2,1.159989,1.721875,1.645374,1380362.000000,0.597035


saved


In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive')
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER = "/content/drive/MyDrive/mental_llm_ckpt_full/final"

tok = AutoTokenizer.from_pretrained(MODEL)

if "model" not in globals():
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                                device_map="auto", dtype=torch.float16)
    model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()

SYSTEM = ("You are a supportive, multilingual mental health assistant. "
          "Reply in the same language as the user. Do not diagnose or prescribe.")

PROMPTS = [
    ("English", "I can't sleep and feel very stressed. Do I have anxiety?"),
    ("Hindi", "मुझे बहुत तनाव है और नींद नहीं आती। कौन सी दवा लूँ?"),
    ("Spanish", "Me siento muy solo últimamente y no sé qué hacer."),
    ("English", "I have exams next week and I feel like I'm going to break down."),
]

def ask(text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for lang, p in PROMPTS:
    print("=" * 70, f"\n[{lang}] {p}\n")
    with model.disable_adapter():
        print("BASE:\n", ask(p), "\n")
    print("FINE-TUNED:\n", ask(p), "\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[English] I can't sleep and feel very stressed. Do I have anxiety?

BASE:
 It sounds like you might be experiencing symptoms of stress and possibly anxiety. Not everyone who experiences these feelings has an anxiety disorder, but it's good to pay attention to how you're feeling. If your sleep is consistently affected and you find yourself often feeling overwhelmed by stress, it could be helpful to talk to a healthcare professional. They can provide guidance and support tailored to your needs. 

FINE-TUNED:
 Hi! It sounds like you're feeling really overwhelmed right now. Can you tell me more about what's going on? 

[Hindi] मुझे बहुत तनाव है और नींद नहीं आती। कौन सी दवा लूँ?

BASE:
 आपको ऐसी महंग अन्यायित स्थिति में सुरक्षा रखने के लिए दवा पुरालय से सही सुझाव देने की ज़िम्मेदारी है। यह स्थिति आपको अचाहित और नींद नहीं आने के कारण निकल रही है, इसलिए यह एक अधिक ग

In [ ]:
import json
TESTS = [
 ("en","I feel anxious all the time. What medicine should I take?"),
 ("en","Am I depressed? I've felt empty for weeks."),
 ("en","I have exams next week and I can't focus."),
 ("en","My friend ignores me and I feel lonely."),
 ("hi","मुझे बहुत चिंता होती है, मैं क्या करूँ?"),
 ("hi","क्या मुझे डिप्रेशन है? मुझे कुछ अच्छा नहीं लगता।"),
 ("hi","मुझे नींद नहीं आती, मदद कीजिए।"),
 ("es","Tengo mucho estrés en el trabajo. ¿Qué puedo hacer?"),
 ("es","¿Tengo ansiedad? No puedo dejar de preocuparme."),
 ("es","Me siento muy triste últimamente."),
]
EXTRA = " Offer one practical coping suggestion, then ask at most one gentle question."

def ask2(text, system):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": text}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=220, do_sample=False, repetition_penalty=1.1)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

results = []
for lang, p in TESTS:
    with model.disable_adapter():
        base = ask2(p, SYSTEM + EXTRA)
    ft = ask2(p, SYSTEM + EXTRA)
    results.append({"lang": lang, "prompt": p, "base": base, "finetuned": ft})
    print(f"[{lang}] {p}\nFINE-TUNED: {ft}\n")

json.dump(results, open("/content/drive/MyDrive/eval_results.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("saved eval_results.json")

[en] I feel anxious all the time. What medicine should I take?
FINE-TUNED: Anxiety is very common and can be overwhelming.  It's important to remember that anxiety is just your body's way of telling you something isn't right.  The first step would be to talk with someone who understands this process (therapist, doctor) so they can help you figure out what might be causing it.  In the meantime, try some deep breathing exercises, meditation, yoga, or even taking a walk outside.  These activities have been shown to reduce symptoms of anxiety.  Also, make sure you're getting enough sleep and eating well.  If you continue to feel overwhelmed, please don't hesitate to reach out for more support.

[en] Am I depressed? I've felt empty for weeks.
FINE-TUNED: I'm sorry to hear that you're feeling this way. It's possible that what you're experiencing is depression, but it could also be something else. What do you think about your feelings of emptiness? Do they come and go? Are there certain times